# Silver Layer — Sessions
Clean session data from Bronze and write standardized Silver table.

## 1. Load bronze table 

In [0]:
df = spark.table("shoply_analytics.bronze.sessions")

## 2. Deduplicate session_id

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window = Window.partitionBy("session_id").orderBy(col("session_id"))

df_clean = (
    df.withColumn("rn", row_number().over(window))
      .filter("rn = 1")
      .drop("rn")
)


## 3. Write to Silver

In [0]:
df_clean.write.format("delta").mode("overwrite").saveAsTable( "shoply_analytics.silver.sessions" )